# Trabalho 1 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências e Carregamento de Dataset

In [1]:
%%capture
pip install NLTK numpy pandas ir_datasets

In [2]:
import ir_datasets

dataset = ir_datasets.load("cranfield")

## 1- Pré-Processamento

In [10]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stemmer = PorterStemmer()


def tokenization(text):
  tokenizer = RegexpTokenizer(r'\w+')
  clean_tokens = tokenizer.tokenize(text)
  return lower_case_normalization(clean_tokens)

def remove_stopwords(words):
  stopwords_set = set(stopwords.words('english'))
  filtered_words = [word for word in words if word not in stopwords_set]
  return filtered_words

def lower_case_normalization(words):
  normalized_words = [word.lower() for word in words]
  return normalized_words

def stemming(words):
  stemmed_words = [stemmer.stem(word) for word in words]
  return stemmed_words

[nltk_data] Downloading package stopwords to /home/arthur/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
def preprocess(text, config_type=0):
  words = tokenization(text)

  if config_type == 1:
    words = remove_stopwords(words)
  if config_type == 2:
    words = stemming(words)
  if config_type == 3:
    words = remove_stopwords(words)
    words = stemming(words)

  return words

## Índice Invertido

In [4]:
class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.document_length = {}
        self.n_documents = 0
        self.avgdl = 0.0

    def build(self, documents, preprocessing_type=0):
        for doc in documents:
            doc_id = doc[0]
            doc_title = doc[1]
            doc_text = doc[2]
            doc_author = doc[3]

            self.n_documents += 1

            title_words = preprocess(doc_title, preprocessing_type)
            text_words = preprocess(doc_text, preprocessing_type)
            author_words = preprocess(doc_author, preprocessing_type)

            all_words = title_words + text_words + author_words

            self.document_length[doc_id] = len(all_words)

            for word in all_words:
                if word not in self.index:
                    self.index[word] = {}

                if doc_id not in self.index[word]:
                    self.index[word][doc_id] = 0

                self.index[word][doc_id] += 1

        self.avgdl = (
            sum(self.document_length.values())
            / len(self.document_length)
        )

    def get_term_frequency(self, term):
        if term in self.index:
            return sum(self.index[term].values())
        else:
            return 0

    def get_k_most_frequent(self, k):
        word_frequencies = {word: self.get_term_frequency(word) for word in self.index}
        sorted_words = sorted(word_frequencies.items(), key=lambda x: x[1], reverse=True)
        return sorted_words[:k]

## 2- Modelo Vetorial

In [5]:
from collections import Counter

class ModeloVetorial:
    def __init__(self, inverted_index):
        self.inverted_index = inverted_index

        # Matriz de pesos: weights[termo][documento] = peso TF-IDF
        self.weights = {}

        # Norma de cada vetor de documento
        self.document_norm = {}

        N = self.inverted_index.n_documents

        for word in self.inverted_index.index:
            self.weights[word] = {}

            ni = len(self.inverted_index.index[word])
            idf = math.log(N / ni)

            for doc_id, fi_j in self.inverted_index.index[word].items():
                wi_j = (1 + math.log(fi_j)) * idf

                # Guarda o peso TF-IDF na matriz
                self.weights[word][doc_id] = wi_j

                if doc_id not in self.document_norm:
                    self.document_norm[doc_id] = 0.0

                self.document_norm[doc_id] += wi_j ** 2

        # Finaliza o cálculo da norma dos documentos
        for doc_id in self.document_norm:
            self.document_norm[doc_id] = math.sqrt(
                self.document_norm[doc_id]
            )

    def score(self, query, doc_id, preprocessing_type=0):
        query_words = preprocess(query, preprocessing_type)
        query_frequency = Counter(query_words)

        query_weights = {}
        product = 0.0
        query_norm = 0.0

        N = self.inverted_index.n_documents

        for word, fi_q in query_frequency.items():
            if word not in self.inverted_index.index:
                continue

            ni = len(self.inverted_index.index[word])
            idf = math.log(N / ni)

            wi_q = (1 + math.log(fi_q)) * idf

            # Guarda o peso TF-IDF do termo na consulta
            query_weights[word] = wi_q

            query_norm += wi_q ** 2

            # Busca o peso do documento diretamente na matriz
            wi_j = self.weights[word].get(doc_id, 0.0)

            product += wi_j * wi_q

        query_norm = math.sqrt(query_norm)
        document_norm = self.document_norm.get(doc_id, 0.0)

        if query_norm == 0 or document_norm == 0:
            return 0.0

        similarity = product / (document_norm * query_norm)

        return similarity

## 3- Modelo Probabilístico (BM25)

In [6]:
import math

class BM25:
    def __init__(self, inverted_index, k1=0.5, b=0):
        self.inverted_index = inverted_index
        self.k1 = k1
        self.b = b

    def score(self, query, doc_id, preprocessing_type=0):
        score = 0.0
        query_words = preprocess(query, preprocessing_type)

        for word in query_words:
            if word in self.inverted_index.index and doc_id in self.inverted_index.index[word]:
                tf = self.inverted_index.index[word][doc_id]
                df = len(self.inverted_index.index[word])
                idf = math.log(1 + ((self.inverted_index.n_documents - df + 0.5) / (df + 0.5)))
                dl = self.inverted_index.document_length[doc_id]
                avgdl = self.inverted_index.avgdl
                score += idf * ((tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * (dl / avgdl))))

        return score

## 4- Métricas de Avaliação

In [7]:
def precision_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = [doc for doc in retrieved_k if doc in relevant_docs]
    precision = len(relevant_retrieved) / k
    return precision

def recall_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = [doc for doc in retrieved_k if doc in relevant_docs]
    recall = len(relevant_retrieved) / len(relevant_docs) if relevant_docs else 0
    return recall

def AP(retrieved_docs, relevant_docs):
    relevant_retrieved = [doc for doc in retrieved_docs if doc in relevant_docs]
    if not relevant_retrieved:
        return 0.0

    precision_sum = 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_docs:
            precision_sum += precision_at_k(retrieved_docs, relevant_docs, i + 1)

    average_precision = precision_sum / len(relevant_retrieved)
    return average_precision

## Rodando Modelos

In [8]:
# Para cada query, armazena em sets os documentos relevantes (grau de relevância >= 1) de qrels

relevant_docs_per_query = {}

for qrel in dataset.qrels_iter():
    query_id = qrel[0]
    doc_id = qrel[1]
    relevance_grade = qrel[2]

    if relevance_grade >= 1:
        if query_id not in relevant_docs_per_query:
            relevant_docs_per_query[query_id] = set()
        relevant_docs_per_query[query_id].add(doc_id)

In [ ]:
preprocessing_type = 3
k = 10

inverted_index = InvertedIndex()
inverted_index.build(
    dataset.docs_iter(),
    preprocessing_type=preprocessing_type
)

bm25 = BM25(inverted_index, 1.2, 0.75)
modelo_vetorial = ModeloVetorial(inverted_index)

metrics_bm25 = {}
metrics_vetorial = {}

retrieved_docs_bm25_per_query = {}
retrieved_docs_vetorial_per_query = {}

for query in dataset.queries_iter():
    query_id = query[0]
    query_text = query[1]

    scores_bm25 = {}
    scores_vetorial = {}

    for doc in dataset.docs_iter():
        doc_id = doc[0]

        score_bm25 = bm25.score(
            query_text,
            doc_id,
            preprocessing_type=preprocessing_type
        )

        scores_bm25[doc_id] = score_bm25

        score_vetorial = modelo_vetorial.score(
            query_text,
            doc_id,
            preprocessing_type=preprocessing_type
        )

        scores_vetorial[doc_id] = score_vetorial


    retrieved_docs_bm25 = sorted(
        scores_bm25,
        key=scores_bm25.get,
        reverse=True
    )

    retrieved_docs_bm25_per_query[query_id] = retrieved_docs_bm25

    retrieved_docs_vetorial = sorted(
        scores_vetorial,
        key=scores_vetorial.get,
        reverse=True
    )

    retrieved_docs_vetorial_per_query[query_id] = retrieved_docs_vetorial

    metrics_bm25[query_id] = [
        precision_at_k(
            retrieved_docs_bm25,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        recall_at_k(
            retrieved_docs_bm25,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        AP(
            retrieved_docs_bm25,
            relevant_docs_per_query.get(query_id, set())
        )
    ]

    metrics_vetorial[query_id] = [
        precision_at_k(
            retrieved_docs_vetorial,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        recall_at_k(
            retrieved_docs_vetorial,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        AP(
            retrieved_docs_vetorial,
            relevant_docs_per_query.get(query_id, set())
        )
    ]

Query ID: 64
Scores BM25: {'1': 0.0, '2': 0.0, '3': 0.0, '4': 0.0, '5': 2.9999449054444915, '6': 1.1906604567418415, '7': 0.0, '8': 0.0, '9': 3.4153524083580837, '10': 0.0, '11': 4.373345632192997, '12': 0.0, '13': 0.0, '14': 9.505031052455081, '15': 9.060618804369774, '16': 2.439458918890604, '17': 0.0, '18': 0.0, '19': 0.0, '20': 3.9621084649711125, '21': 0.0, '22': 2.5923522427109793, '23': 0.0, '24': 2.018043083587821, '25': 9.221736613084992, '26': 0.0, '27': 4.3476348559501155, '28': 0.0, '29': 0.0, '30': 2.1923673995084965, '31': 0.0, '32': 2.667135458197313, '33': 1.947906817426812, '34': 2.2809096946541483, '35': 1.0426480592314502, '36': 3.614853867195202, '37': 11.345317324331765, '38': 3.4536945399161603, '39': 0.0, '40': 2.095219494041663, '41': 1.278478411494968, '42': 10.418338454207126, '43': 0.0, '44': 8.753250839064071, '45': 3.7956119361721394, '46': 0.0, '47': 7.267936314331932, '48': 3.388490819207393, '49': 4.988932090112944, '50': 1.682053100886513, '51': 0.0, '5

In [ ]:
print("Métricas do BM25:")
print(metrics_bm25)

print("\nMétricas do Modelo Vetorial:")
print(metrics_vetorial)

Métricas do BM25:
{'1': [0.3, 0.10714285714285714, 0.2254130367325213], '2': [0.5, 0.20833333333333334, 0.23301067676852075], '3': [0.7, 0.875, 0.6588010204081634], '4': [0.1, 0.5, 0.5303030303030303], '5': [0.3, 0.75, 0.41754385964912283], '6': [0.1, 0.25, 0.16999999999999998], '7': [0.2, 0.4, 0.17789766970618037], '8': [0.1, 0.09090909090909091, 0.15964195135145398], '9': [0.3, 1.0, 0.9166666666666666], '10': [0.1, 0.125, 0.1440389440531494], '11': [0.2, 0.2857142857142857, 0.1925551975176035], '12': [0.2, 0.4, 0.27954498977505116], '13': [0.0, 0.0, 0.007836726087423283], '14': [0.1, 0.5, 0.5909090909090909], '15': [0.2, 1.0, 1.0], '16': [0.1, 0.3333333333333333, 0.19751046486010135], '17': [0.1, 0.5, 0.10704225352112677], '18': [0.1, 0.3333333333333333, 0.35908626248329006], '19': [0.1, 0.1111111111111111, 0.07660946471859176], '20': [0.5, 0.5555555555555556, 0.5586089654717106], '21': [0.0, 0.0, 0.060952395162921474], '22': [0.0, 0.0, 0.001644736842105263], '23': [0.5, 0.15625, 0.2

## 5- Comparação entre modelos

Objetivo: calcular a média das métricas para cada modelo. Em seguida, analisar as consultas com maiores divergências entre os modelos para compreender o porquê de tais diferenças

In [ ]:
# Armazena queries num dicionário

query_dict = {}

for query in dataset.queries_iter():
    query_dict[query[0]] = query[1]

In [ ]:
#Rodar os modelos e avaliar em quais consultas ocorreram as maiores divergências e hipóteses do porquê (Arthur)

TOTAL_PRECISION_BM25 = sum([metrics_bm25[query_id][0] for query_id in metrics_bm25]) / len(metrics_bm25)
TOTAL_PRECISION_VETORIAL = sum([metrics_vetorial[query_id][0] for query_id in metrics_vetorial]) / len(metrics_vetorial)

TOTAL_RECALL_BM25 = sum([metrics_bm25[query_id][1] for query_id in metrics_bm25]) / len(metrics_bm25)
TOTAL_RECALL_VETORIAL = sum([metrics_vetorial[query_id][1] for query_id in metrics_vetorial]) / len(metrics_vetorial)

MAP_BM25 = sum([metrics_bm25[query_id][2] for query_id in metrics_bm25]) / len(metrics_bm25)
MAP_VETORIAL = sum([metrics_vetorial[query_id][2] for query_id in metrics_vetorial]) / len(metrics_vetorial)

print(TOTAL_PRECISION_BM25, TOTAL_RECALL_BM25, MAP_BM25)
print(TOTAL_PRECISION_VETORIAL, TOTAL_RECALL_VETORIAL, MAP_VETORIAL)

0.23955555555555555 0.40568523368807163 0.31741839569031133
0.23600000000000002 0.4004241069292669 0.29919253940319546


In [ ]:
def queries_biggest_difference(metrics_bm25, metrics_vetorial):
    differences = {}

    for query_id in metrics_bm25:
        precision_bm25 = metrics_bm25[query_id][0]
        precision_vetorial = metrics_vetorial[query_id][0]

        recall_bm25 = metrics_bm25[query_id][1]
        recall_vetorial = metrics_vetorial[query_id][1]

        ap_bm25 = metrics_bm25[query_id][2]
        ap_vetorial = metrics_vetorial[query_id][2]

        precision_diff = abs(precision_bm25 - precision_vetorial)
        recall_diff = abs(recall_bm25 - recall_vetorial)
        ap_diff = abs(ap_bm25 - ap_vetorial)

        total_diff = precision_diff + recall_diff + ap_diff

        differences[query_id] = total_diff

    sorted_differences = sorted(differences.items(), key=lambda x: x[1], reverse=True)

    return sorted_differences

print("\nTop-3 Consultas com maiores divergências entre BM25 e Vetorial: \n")
top3_differences = queries_biggest_difference(metrics_bm25, metrics_vetorial)[:3]

for query_id, diff in top3_differences:
    print(f"Query ID: {query_id}, Diferença Total: {diff}")
    print(f"Precision BM25: {metrics_bm25[query_id][0]}, Precision Vetorial: {metrics_vetorial[query_id][0]}")
    print(f"Recall BM25: {metrics_bm25[query_id][1]}, Recall Vetorial: {metrics_vetorial[query_id][1]}")
    print(f"AP BM25: {metrics_bm25[query_id][2]}, AP Vetorial: {metrics_vetorial[query_id][2]}")
    print(f"Query Text: {query_dict[query_id]}\n")


Top-3 Consultas com maiores divergências entre BM25 e Vetorial: 

Query ID: 190, Diferença Total: 1.090958605664488
Precision BM25: 0.2, Precision Vetorial: 0.5
Recall BM25: 0.4, Recall Vetorial: 1.0
AP BM25: 0.44237472766884534, AP Vetorial: 0.6333333333333333
Query Text: will an analysis of panel flutter based on arbitrarily assumed modes of
deformation prove satisfactory,  and if so,  what is the minimum number
of modes that need be considered .

Query ID: 167, Diferença Total: 0.961111111111111
Precision BM25: 0.2, Precision Vetorial: 0.1
Recall BM25: 1.0, Recall Vetorial: 0.5
AP BM25: 0.5, AP Vetorial: 0.1388888888888889
Query Text: exact solution methods for calculating the ablative mass loss of a
material ablating at high temperatures in a hypersonic flight
environment .

Query ID: 27, Diferença Total: 0.9195205815895472
Precision BM25: 0.2, Precision Vetorial: 0.0
Recall BM25: 0.6666666666666666, Recall Vetorial: 0.0
AP BM25: 0.13893450100346652, AP Vetorial: 0.086080586080586

In [ ]:
for query_id, diff in top3_differences:
    query_relevant_docs = relevant_docs_per_query.get(query_id, set())
    
    print(f"\n\n Query_ID: {query_id}")
    print(retrieved_docs_bm25_per_query[query_id][:10])
    print(retrieved_docs_vetorial_per_query[query_id][:10])
    print(f"Relevant Docs: {query_relevant_docs}")



 Query_ID: 190
['390', '627', '15', '856', '1339', '52', '685', '766', '1008', '14']
['390', '627', '391', '856', '1008', '15', '1111', '285', '766', '864']
Relevant Docs: {'864', '285', '15', '390', '391'}


 Query_ID: 167
['1279', '274', '553', '82', '1098', '1065', '1096', '1101', '1100', '1043']
['553', '1279', '1099', '1100', '1098', '1241', '1065', '1097', '274', '1096']
Relevant Docs: {'82', '274'}


 Query_ID: 27
['1035', '1031', '1176', '888', '512', '1178', '428', '1293', '224', '1129']
['1176', '888', '1031', '838', '1129', '1178', '1133', '512', '1293', '1035']
Relevant Docs: {'278', '224', '428'}


## 6- Análise por consulta

Objetivo: identificar 2 consultas para cada uma das seguintes situações: BM25 superior ao vetorial, vetorial superior ao BM25 e BM25 e vetorial com desempenhos insatisfatórios

In [ ]:
#Identificar 2 consultas que BM25 > vetorial, 2 em que vetorial > BM25 e 2 em que ambos foram ruins (Arthur)

top20_differences = queries_biggest_difference(metrics_bm25, metrics_vetorial)[:20]

for query_id, diff in top20_differences:
    print(f"Query ID: {query_id}, Diferença Total: {diff}")
    print(f"Precision BM25: {metrics_bm25[query_id][0]}, Precision Vetorial: {metrics_vetorial[query_id][0]}")
    print(f"Recall BM25: {metrics_bm25[query_id][1]}, Recall Vetorial: {metrics_vetorial[query_id][1]}")
    print(f"AP BM25: {metrics_bm25[query_id][2]}, AP Vetorial: {metrics_vetorial[query_id][2]}")
    print('')


Query ID: 190, Diferença Total: 1.090958605664488
Precision BM25: 0.2, Precision Vetorial: 0.5
Recall BM25: 0.4, Recall Vetorial: 1.0
AP BM25: 0.44237472766884534, AP Vetorial: 0.6333333333333333

Query ID: 167, Diferença Total: 0.961111111111111
Precision BM25: 0.2, Precision Vetorial: 0.1
Recall BM25: 1.0, Recall Vetorial: 0.5
AP BM25: 0.5, AP Vetorial: 0.1388888888888889

Query ID: 27, Diferença Total: 0.9195205815895472
Precision BM25: 0.2, Precision Vetorial: 0.0
Recall BM25: 0.6666666666666666, Recall Vetorial: 0.0
AP BM25: 0.13893450100346652, AP Vetorial: 0.08608058608058607

Query ID: 21, Diferença Total: 0.8366153879311774
Precision BM25: 0.0, Precision Vetorial: 0.2
Recall BM25: 0.0, Recall Vetorial: 0.5
AP BM25: 0.060952395162921474, AP Vetorial: 0.1975677830940989

Query ID: 95, Diferença Total: 0.8340909090909091
Precision BM25: 0.2, Precision Vetorial: 0.1
Recall BM25: 1.0, Recall Vetorial: 0.5
AP BM25: 0.45, AP Vetorial: 0.2159090909090909

Query ID: 120, Diferença Tota

In [ ]:
def get_bad_queries(metrics_bm25, metrics_vetorial):
    bad_queries = []

    for query_id in metrics_bm25:
        precision_bm25 = metrics_bm25[query_id][0]
        precision_vetorial = metrics_vetorial[query_id][0]

        recall_bm25 = metrics_bm25[query_id][1]
        recall_vetorial = metrics_vetorial[query_id][1]

        ap_bm25 = metrics_bm25[query_id][2]
        ap_vetorial = metrics_vetorial[query_id][2]

        if (precision_bm25 <= 0.1 and precision_vetorial <= 0.1) and (recall_bm25 <= 0.1 and recall_vetorial <= 0.1) and (ap_bm25 <= 0.1 and ap_vetorial <= 0.1):
            bad_queries.append(query_id)

    return bad_queries

bad_queries = get_bad_queries(metrics_bm25, metrics_vetorial)

for query in bad_queries:
    print(f"Query ID: {query}\n Precision BM25: {metrics_bm25[query][0]}, Precision Vetorial: {metrics_vetorial[query][0]}")
    print(f"Recall BM25: {metrics_bm25[query][1]}, Recall Vetorial: {metrics_vetorial[query][1]}")
    print(f"AP BM25: {metrics_bm25[query][2]}, AP Vetorial: {metrics_vetorial[query][2]}")
    print('')

Query ID: 13
 Precision BM25: 0.0, Precision Vetorial: 0.0
Recall BM25: 0.0, Recall Vetorial: 0.0
AP BM25: 0.007836726087423283, AP Vetorial: 0.007836726087423283

Query ID: 22
 Precision BM25: 0.0, Precision Vetorial: 0.0
Recall BM25: 0.0, Recall Vetorial: 0.0
AP BM25: 0.001644736842105263, AP Vetorial: 0.001644736842105263

Query ID: 28
 Precision BM25: 0.0, Precision Vetorial: 0.0
Recall BM25: 0.0, Recall Vetorial: 0.0
AP BM25: 0.004906083543594057, AP Vetorial: 0.005056980056980057

Query ID: 31
 Precision BM25: 0.0, Precision Vetorial: 0.0
Recall BM25: 0.0, Recall Vetorial: 0.0
AP BM25: 0.0009132420091324201, AP Vetorial: 0.0009407337723424271

Query ID: 35
 Precision BM25: 0.0, Precision Vetorial: 0.0
Recall BM25: 0.0, Recall Vetorial: 0.0
AP BM25: 0.034126147357975686, AP Vetorial: 0.04691640626892426

Query ID: 38
 Precision BM25: 0.0, Precision Vetorial: 0.0
Recall BM25: 0.0, Recall Vetorial: 0.0
AP BM25: 0.05029928630967849, AP Vetorial: 0.03141428892181106

Query ID: 44
 Pre

Queries escolhidas para o tópico:

- **Queries nas quais BM25 é superior ao Vetorial**: 27, 195

- **Queries nas quais Vetorial é superior ao BM25**: 190, 21

- **Queries nas quais BM25 e Vetorial são ruins**: 13, 28

In [ ]:
queries_chosen = [["27", "195"], ["190", "21"], ["13", "28"]]

for subarray in queries_chosen:
    for query_id in subarray:
        relevant_docs = relevant_docs_per_query.get(query_id, set())
        bm25_top10 = retrieved_docs_bm25_per_query[query_id][:10]
        vetorial_top10 = retrieved_docs_vetorial_per_query[query_id][:10]
        
        BM25_intersection = list(
            set(bm25_top10) & set(relevant_docs)
        )

        Vetorial_intersection = list(
            set(vetorial_top10) & set(relevant_docs)
        )

        print(f"Query_ID: {query_id}")
        print(f"Documentos Relevantes da Query: {relevant_docs_per_query.get(query_id, set())}")
        print(f"Top-10 Documentos Recuperados pelo BM25: {bm25_top10}")
        print(f"Top-10 Documentos Recuperados pelo Vetorial: {vetorial_top10}")
        print(f"Documentos Relevantes e Recuperados (BM25): {BM25_intersection}")
        print(f"Documentos Relevantes e Recuperados (Vetorial): {Vetorial_intersection}")
        print('')
    print('\n-------------------------------------------------------\n')

Query_ID: 27
Documentos Relevantes da Query: {'428', '278', '224'}
Top-10 Documentos Recuperados pelo BM25: ['1035', '1031', '1176', '888', '512', '1178', '428', '1293', '224', '1129']
Top-10 Documentos Recuperados pelo Vetorial: ['1176', '888', '1031', '838', '1129', '1178', '1133', '512', '1293', '1035']
Documentos Relevantes e Recuperados (BM25): ['428', '224']
Documentos Relevantes e Recuperados (Vetorial): []

Query_ID: 195
Documentos Relevantes da Query: {'739', '742', '743'}
Top-10 Documentos Recuperados pelo BM25: ['642', '744', '1055', '739', '932', '831', '1173', '1126', '740', '769']
Top-10 Documentos Recuperados pelo Vetorial: ['642', '932', '1055', '744', '1054', '888', '842', '457', '831', '1045']
Documentos Relevantes e Recuperados (BM25): ['739']
Documentos Relevantes e Recuperados (Vetorial): []


-------------------------------------------------------

Query_ID: 190
Documentos Relevantes da Query: {'15', '390', '864', '285', '391'}
Top-10 Documentos Recuperados pelo B

## 7- Variação dos parâmetros do BM25

In [ ]:
#Impacto na mudança de parâmetros do BM25 (Arthur)



## 8- Modificação de consultas

In [ ]:
# @title
import pandas as pd
from IPython.display import display

# Normaliza os IDs porque algumas versões do Cranfield usam
# preenchimento com zeros, como "057", enquanto outras usam "57".
todas_as_consultas = {
    str(int(str(query[0]).strip())): {
        "id_dataset": query[0],
        "texto": query[1]
    }
    for query in dataset.queries_iter()
}

print(f"Total de consultas no Cranfield: {len(todas_as_consultas)}")


# Guarda os dados dos documentos para evitar percorrer o dataset
# repetidamente durante a exibição dos resultados.
documentos_cranfield = {
    doc[0]: {
        "titulo": doc[1],
        "texto": doc[2],
        "autores": doc[3]
    }
    for doc in dataset.docs_iter()
}


pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

def resumir_campo(valor, limite):
    """Remove quebras de linha e abrevia um campo para exibição."""
    valor = " ".join(str(valor).split())

    if len(valor) <= limite:
        return valor

    return valor[:limite - 3] + "..."


def gerar_top10(texto_consulta, modelo, query_id):
    """
    Calcula o ranking de uma consulta e devolve os dez primeiros
    documentos em um DataFrame.
    """
    pontuacoes = {
        doc_id: modelo.score(
            texto_consulta,
            doc_id,
            preprocessing_type=preprocessing_type
        )
        for doc_id in documentos_cranfield
    }

    ranking = sorted(
        pontuacoes.items(),
        key=lambda item: (-item[1], int(item[0]))
    )[:10]

    # O julgamento de relevância continua associado ao par:
    # (query_id, doc_id).
    id_dataset = todas_as_consultas[query_id]["id_dataset"]

    relevantes = (
        relevant_docs_per_query.get(id_dataset, set())
        | relevant_docs_per_query.get(query_id, set())
    )

    linhas = [
        {
            "Posição": posicao,
            "Doc.": doc_id,
            "Score": round(score, 6),
            "Relevante": "Sim" if doc_id in relevantes else "Não",
            "Título": resumir_campo(
                documentos_cranfield[doc_id]["titulo"],
                100
            ),
            "Autores": resumir_campo(
                documentos_cranfield[doc_id]["autores"],
                70
            ),
            "Texto": resumir_campo(
                documentos_cranfield[doc_id]["texto"],
                240
            )
        }
        for posicao, (doc_id, score) in enumerate(
            ranking,
            start=1
        )
    ]

    return pd.DataFrame(linhas)


def exibir_mudancas_top10(
    top10_original,
    top10_alternativa,
    nome_modelo
):
    """
    Mostra quais documentos permaneceram, entraram ou saíram
    do Top-10 após a modificação da consulta.
    """
    docs_original = top10_original["Doc."].tolist()
    docs_alternativa = top10_alternativa["Doc."].tolist()

    permaneceram = [
        doc_id
        for doc_id in docs_alternativa
        if doc_id in docs_original
    ]

    entraram = [
        doc_id
        for doc_id in docs_alternativa
        if doc_id not in docs_original
    ]

    sairam = [
        doc_id
        for doc_id in docs_original
        if doc_id not in docs_alternativa
    ]

    print(f"\nMudanças no Top-10 — {nome_modelo}")
    print(f"Documentos mantidos: {permaneceram}")
    print(f"Documentos que entraram: {entraram}")
    print(f"Documentos que saíram: {sairam}")
    print(f"Sobreposição: {len(permaneceram)}/10")


def exibir_comparacao(
    numero,
    query_id,
    consulta_original,
    consulta_alternativa,
    tipo_modificacao,
    detalhes_modificacao
):
    """
    Executa a consulta original e a alternativa nos dois modelos,
    exibindo as modificações, os rankings e as mudanças no Top-10.
    """
    if query_id not in todas_as_consultas:
        raise ValueError(
            f"A consulta {query_id} não existe no Cranfield."
        )

    print("\n" + "=" * 120)
    print(f"CONSULTA {numero} | ID Cranfield: {query_id}")
    print(f"Modificações utilizadas: {tipo_modificacao}")

    print("\nDETALHES DAS MODIFICAÇÕES:")
    for detalhe in detalhes_modificacao:
        print(f"- {detalhe}")

    print("\nCONSULTA ORIGINAL:")
    print(consulta_original)

    print("\nCONSULTA ALTERNATIVA:")
    print(consulta_alternativa)

    # Modelo Vetorial
    top10_vetorial_original = gerar_top10(
        consulta_original,
        modelo_vetorial,
        query_id
    )

    top10_vetorial_alternativa = gerar_top10(
        consulta_alternativa,
        modelo_vetorial,
        query_id
    )

    print("\nMODELO VETORIAL — CONSULTA ORIGINAL")
    display(top10_vetorial_original)

    print("\nMODELO VETORIAL — CONSULTA ALTERNATIVA")
    display(top10_vetorial_alternativa)

    exibir_mudancas_top10(
        top10_vetorial_original,
        top10_vetorial_alternativa,
        "Modelo Vetorial"
    )

    # BM25
    top10_bm25_original = gerar_top10(
        consulta_original,
        bm25,
        query_id
    )

    top10_bm25_alternativa = gerar_top10(
        consulta_alternativa,
        bm25,
        query_id
    )

    print("\nBM25 — CONSULTA ORIGINAL")
    display(top10_bm25_original)

    print("\nBM25 — CONSULTA ALTERNATIVA")
    display(top10_bm25_alternativa)

    exibir_mudancas_top10(
        top10_bm25_original,
        top10_bm25_alternativa,
        "BM25"
    )


# ============================================================
# CONSULTA 1
# ============================================================

consulta_1_id = "57"
consulta_1_original = todas_as_consultas[consulta_1_id]["texto"]

consulta_1_alternativa = (
    "what steady and non-steady aerodynamic flow characteristics "
    "affect the wing flutter mechanism?"
)

exibir_comparacao(
    numero=1,
    query_id=consulta_1_id,
    consulta_original=consulta_1_original,
    consulta_alternativa=consulta_1_alternativa,
    tipo_modificacao=(
        "Remover termos + acrescentar termos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Remover termos: foi removido o termo genérico "
            "'significant'."
        ),
        (
            "Acrescentar termos: foram acrescentados "
            "'aerodynamic' e 'wing'."
        ),
        (
            "Tornar mais específica: a consulta passou a relacionar "
            "o escoamento aerodinâmico ao flutter de asas."
        )
    ]
)


# ============================================================
# CONSULTA 2
# ============================================================

consulta_2_id = "121"
consulta_2_original = todas_as_consultas[consulta_2_id]["texto"]

consulta_2_alternativa = (
    "what papers deal with circumferential buckling of thin cylindrical "
    "shells due to thermal or mechanical loading?"
)

exibir_comparacao(
    numero=2,
    query_id=consulta_2_id,
    consulta_original=consulta_2_original,
    consulta_alternativa=consulta_2_alternativa,
    tipo_modificacao=(
        "Remover termos + acrescentar termos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Remover termos: foram removidos 'are there' "
            "e a repetição de 'buckling'."
        ),
        (
            "Acrescentar termos: foi acrescentada a expressão "
            "'thin cylindrical shells'."
        ),
        (
            "Tornar mais específica: a consulta foi delimitada "
            "à flambagem de cascas cilíndricas finas."
        )
    ]
)


# ============================================================
# CONSULTA 3
# ============================================================

consulta_3_id = "183"
consulta_3_original = todas_as_consultas[consulta_3_id]["texto"]

consulta_3_alternativa = (
    "what factors, including lift and aircraft geometry, have been shown "
    "to have a primary influence on sonic boom intensity?"
)

exibir_comparacao(
    numero=3,
    query_id=consulta_3_id,
    consulta_original=consulta_3_original,
    consulta_alternativa=consulta_3_alternativa,
    tipo_modificacao=(
        "Acrescentar termos + usar sinônimos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Acrescentar termos: foram acrescentados "
            "'lift' e 'aircraft geometry'."
        ),
        (
            "Usar sinônimos: 'strength' foi substituído "
            "por 'intensity'."
        ),
        (
            "Tornar mais específica: a consulta passou a mencionar "
            "fatores aerodinâmicos relacionados ao sonic boom."
        )
    ]
)


# ============================================================
# CONSULTA 4
# ============================================================

consulta_4_id = "56"
consulta_4_original = todas_as_consultas[consulta_4_id]["texto"]

consulta_4_alternativa = (
    "to what extent can steady-state aerodynamic data be used to "
    "estimate wing flutter characteristics?"
)

exibir_comparacao(
    numero=4,
    query_id=consulta_4_id,
    consulta_original=consulta_4_original,
    consulta_alternativa=consulta_4_alternativa,
    tipo_modificacao=(
        "Acrescentar termos + usar sinônimos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Acrescentar termos: foi acrescentado o termo 'wing'."
        ),
        (
            "Usar sinônimos: 'utilized' foi substituído por 'used' "
            "e 'predict' foi substituído por 'estimate'."
        ),
        (
            "Tornar mais específica: a consulta passou a tratar "
            "das características de flutter de asas."
        )
    ]
)


# ============================================================
# CONSULTA 5
# ============================================================

consulta_5_id = "219"
consulta_5_original = todas_as_consultas[consulta_5_id]["texto"]

consulta_5_alternativa = (
    "what are the effects on viscous flow fields around circular "
    "cylinders when the Reynolds number is low?"
)

exibir_comparacao(
    numero=5,
    query_id=consulta_5_id,
    consulta_original=consulta_5_original,
    consulta_alternativa=consulta_5_alternativa,
    tipo_modificacao=(
        "Acrescentar termos + usar sinônimos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Acrescentar termos: foram acrescentados "
            "'viscous' e 'circular cylinders'."
        ),
        (
            "Usar sinônimos: 'small' foi substituído por 'low'."
        ),
        (
            "Tornar mais específica: a consulta foi delimitada "
            "ao escoamento viscoso ao redor de cilindros circulares."
        )
    ]
)

Total de consultas no Cranfield: 225

CONSULTA 1 | ID Cranfield: 57
Modificações utilizadas: Remover termos + acrescentar termos + tornar mais específica

DETALHES DAS MODIFICAÇÕES:
- Remover termos: foi removido o termo genérico 'significant'.
- Acrescentar termos: foram acrescentados 'aerodynamic' e 'wing'.
- Tornar mais específica: a consulta passou a relacionar o escoamento aerodinâmico ao flutter de asas.

CONSULTA ORIGINAL:
what are the significant steady and non-steady flow characteristics
which affect the flutter mechanism .

CONSULTA ALTERNATIVA:
what steady and non-steady aerodynamic flow characteristics affect the wing flutter mechanism?



MODELO VETORIAL — CONSULTA ORIGINAL


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,753,0.225223,Não,development of a quasi-steady approach to flutter and correlation with kernel-function...,"gravitz,s.i., laidlaw,w.r., bryce,w.w. and cooper,r.e.",development of a quasi-steady approach to flutter and correlation with kernel-function...
1,2,380,0.180592,Sim,effect of quasi-steady air forces on incompressible bending-torsion flutter .,"dugundi,j.",effect of quasi-steady air forces on incompressible bending-torsion flutter . explicit...
2,3,507,0.180087,Não,energy equation approximations in fluid mechanics .,"goldstein,a.w.",energy equation approximations in fluid mechanics . discussion of several forms of the...
3,4,444,0.164631,Não,an approach to the flutter problem in real fluids .,"rott,n. and george,m.b.t.",an approach to the flutter problem in real fluids . an approximate theory of airfoils ...
4,5,1099,0.162654,Não,a theoretical study of stagnation point ablation .,"roberts, l.",a theoretical study of stagnation point ablation . a simplified analysis is made of th...
5,6,1181,0.160962,Não,steady magnetohydrodynamic flow past a non-conducting wedge .,"chu,c.k. and lynn,y.m.",steady magnetohydrodynamic flow past a non-conducting wedge . this paper presents a st...
6,7,52,0.159361,Não,procedure for calculating flutter at high supersonic speed including camber deflection...,"morgan,h.g.",procedure for calculating flutter at high supersonic speed including camber deflection...
7,8,1111,0.158937,Não,some research on high speed flutter .,"garrick, i.e.",some research on high speed flutter . paper presents brief discussions of many topics ...
8,9,1339,0.158082,Não,calculation of flutter characteristics for finite-span swept or unswept wings at subso...,"yates, e.c.",calculation of flutter characteristics for finite-span swept or unswept wings at subso...
9,10,916,0.152987,Não,the flow around oscillating low aspect ratio wings at transonic speeds .,"landahl, m.t.",the flow around oscillating low aspect ratio wings at transonic speeds . when certain ...



MODELO VETORIAL — CONSULTA ALTERNATIVA


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,753,0.256830,Não,development of a quasi-steady approach to flutter and correlation with kernel-function...,"gravitz,s.i., laidlaw,w.r., bryce,w.w. and cooper,r.e.",development of a quasi-steady approach to flutter and correlation with kernel-function...
1,2,52,0.221713,Não,procedure for calculating flutter at high supersonic speed including camber deflection...,"morgan,h.g.",procedure for calculating flutter at high supersonic speed including camber deflection...
2,3,1339,0.208615,Não,calculation of flutter characteristics for finite-span swept or unswept wings at subso...,"yates, e.c.",calculation of flutter characteristics for finite-span swept or unswept wings at subso...
3,4,916,0.186558,Não,the flow around oscillating low aspect ratio wings at transonic speeds .,"landahl, m.t.",the flow around oscillating low aspect ratio wings at transonic speeds . when certain ...
4,5,380,0.181625,Sim,effect of quasi-steady air forces on incompressible bending-torsion flutter .,"dugundi,j.",effect of quasi-steady air forces on incompressible bending-torsion flutter . explicit...
5,6,507,0.181116,Não,energy equation approximations in fluid mechanics .,"goldstein,a.w.",energy equation approximations in fluid mechanics . discussion of several forms of the...
6,7,1111,0.179508,Não,some research on high speed flutter .,"garrick, i.e.",some research on high speed flutter . paper presents brief discussions of many topics ...
7,8,749,0.178501,Não,the aerodynamic effects of aspect ratio and sweepback on wing flutter .,"molyneux,w.g. and hall,h.",the aerodynamic effects of aspect ratio and sweepback on wing flutter . the report des...
8,9,1338,0.167018,Não,investigation to determine effects of center of gravity location on the transonic flut...,"jones, g.w. and unangst, j.r.",investigation to determine effects of center of gravity location on the transonic flut...
9,10,391,0.165939,Não,flutter of rectangular simply supported panels at high supersonic speeds .,"hedgepeth,j.m.",flutter of rectangular simply supported panels at high supersonic speeds . the problem...


...


Mudanças no Top-10 — Modelo Vetorial
Documentos mantidos: []
Documentos que entraram: ['1084', '1081', '1078', '1258', '533', '1105', '1253', '483', '666', '382']
Documentos que saíram: ['1221', '1222', '208', '992', '775', '342', '530', '1309', '668', '1377']
Sobreposição: 0/10

BM25 — CONSULTA ORIGINAL


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,1221,12.596137,Não,"steady flow of conducting fluids in channels under transverse magnetic fields, with co...","tani,i.","steady flow of conducting fluids in channels under transverse magnetic fields, with co..."
1,2,208,11.673052,Não,the hall effect in the viscous flow of ionized gas between parallel plates under trans...,"sato,h.",the hall effect in the viscous flow of ionized gas between parallel plates under trans...
2,3,1222,11.146776,Não,axisymmetric magnetohydrodynamic channel flow .,"hains,f.d. and holer,y.a.",axisymmetric magnetohydrodynamic channel flow . the axisymmetric subsonic and superson...
3,4,992,10.614028,Não,the effects of a small jet of air exhausting from the nose of a body of revolution in ...,"love, e.s.",the effects of a small jet of air exhausting from the nose of a body of revolution in ...
4,5,993,10.410687,Não,the extent of the jet interference flow fields . jet effects on cylindrical afterbodie...,"hayman, l.o. and mcdearmon, r.w.",the extent of the jet interference flow fields . jet effects on cylindrical afterbodie...
5,6,299,9.877386,Não,magnetohydrodynamic flow past a semi-infinite plate .,"meksyn,d.",magnetohydrodynamic flow past a semi-infinite plate . the flow of viscous electrically...
6,7,371,9.619219,Não,note on tip-bluntness effects in the supersonic and hypersonic regimes .,"bennett,f.d.",note on tip-bluntness effects in the supersonic and hypersonic regimes . in a recent l...
7,8,530,9.454634,Não,an aerodynamic analysis for flutter in oseen-type viscous flow .,"chu,wen-hwa.",an aerodynamic analysis for flutter in oseen-type viscous flow . oseen's equations for...
8,9,1309,9.048995,Não,hypersonic flows past a yawed circular cone and other pointed bodies .,"cheng,h.k.",hypersonic flows past a yawed circular cone and other pointed bodies . a detailed trea...
9,10,7,9.047126,Não,the effect of controlled three-dimensional roughness on boundary layer transition at s...,"van driest,e.r. and mccauley,w.d.",the effect of controlled three-dimensional roughness on boundary layer transition at s...



BM25 — CONSULTA ALTERNATIVA


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,1081,21.183343,Sim,numerical solution of the navier-stokes equations for the flow around a circular cylin...,"kawaguti, m.",numerical solution of the navier-stokes equations for the flow around a circular cylin...
1,2,1078,17.706204,Sim,the steady flow of a viscous fluid past a circular cylinder at reynolds numbers 40 and...,"apelt, c.j",the steady flow of a viscous fluid past a circular cylinder at reynolds numbers 40 and...
2,3,1084,16.292369,Não,the flow past circular cylinders at low speeds .,"thom, a.",the flow past circular cylinders at low speeds . this paper deals chiefly with calcula...
3,4,1105,15.976758,Não,numerical solutions for supersonic flow of an ideal gas around blunt two-dimensional b...,"fuller,f.b.",numerical solutions for supersonic flow of an ideal gas around blunt two-dimensional b...
4,5,1258,15.336560,Sim,"heat transfer, recovery factor and pressure distributions around a circular cylinder n...","tewfik,o.k. and giedt,w.h.","heat transfer, recovery factor and pressure distributions around a circular cylinder n..."
5,6,1253,14.387606,Não,hypersonic viscous flow near the stagnation point in the presence of magnetic field .,"wu,ching-sheng.",hypersonic viscous flow near the stagnation point in the presence of magnetic field . ...
6,7,1082,14.203060,Não,"the flow past pitot tube at low reynolds numbers, part 1-dash the numerical solution o...","lester, w.g.s.","the flow past pitot tube at low reynolds numbers, part 1-dash the numerical solution o..."
7,8,208,14.023818,Não,the hall effect in the viscous flow of ionized gas between parallel plates under trans...,"sato,h.",the hall effect in the viscous flow of ionized gas between parallel plates under trans...
8,9,371,13.912470,Não,note on tip-bluntness effects in the supersonic and hypersonic regimes .,"bennett,f.d.",note on tip-bluntness effects in the supersonic and hypersonic regimes . in a recent l...
9,10,1395,13.810170,Sim,low density stagnation point heat transfer measurements in the hypersonic shock tunnel .,"wilson,m.r. and wittliff,c.e.",low density stagnation point heat transfer measurements in the hypersonic shock tunnel...



Mudanças no Top-10 — BM25
Documentos mantidos: ['208', '371']
Documentos que entraram: ['1081', '1078', '1084', '1105', '1258', '1253', '1082', '1395']
Documentos que saíram: ['1221', '1222', '992', '993', '299', '530', '1309', '7']
Sobreposição: 2/10


## 9- Análise de erros

In [ ]:
# @title
import math
import pandas as pd
from collections import Counter
from IPython.display import display

QUERIES = ["57"]
TOP_K = 10
TOPO = 5
N_FALSOS_POSITIVOS = 2
N_RELEVANTES_OMITIDOS = 2

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)


def normalizar_id(valor):
    texto = str(valor).strip()
    return str(int(texto)) if texto.isdigit() else texto


def chave_id(valor):
    texto = normalizar_id(valor)
    return (0, int(texto)) if texto.isdigit() else (1, texto)


def resumir(valor, limite):
    texto = " ".join(str(valor).split())
    return texto if len(texto) <= limite else texto[:limite - 3] + "..."


consultas = {
    normalizar_id(query[0]): {
        "id_dataset": query[0],
        "texto": query[1]
    }
    for query in dataset.queries_iter()
}

documentos = {
    normalizar_id(doc[0]): {
        "id_dataset": doc[0],
        "titulo": doc[1],
        "texto": doc[2],
        "autores": doc[3]
    }
    for doc in dataset.docs_iter()
}

relevantes = {}

for qrel in dataset.qrels_iter():
    if qrel[2] >= 1:
        query_id = normalizar_id(qrel[0])
        doc_id = normalizar_id(qrel[1])

        relevantes.setdefault(
            query_id,
            set()
        ).add(doc_id)

modelos = {
    "Modelo Vetorial": modelo_vetorial,
    "BM25": bm25
}

cache_rankings = {}


def calcular_ranking(query_id, nome_modelo):
    query_id = normalizar_id(query_id)
    chave = (query_id, nome_modelo)

    if chave in cache_rankings:
        return cache_rankings[chave]

    texto_query = consultas[query_id]["texto"]
    modelo = modelos[nome_modelo]

    ranking = [
        (
            doc_id,
            modelo.score(
                texto_query,
                dados["id_dataset"],
                preprocessing_type=preprocessing_type
            )
        )
        for doc_id, dados in documentos.items()
    ]

    ranking.sort(
        key=lambda item: (
            -item[1],
            chave_id(item[0])
        )
    )

    cache_rankings[chave] = ranking
    return ranking


def selecionar_erros(query_id, ranking):
    docs_relevantes = relevantes.get(query_id, set())

    falsos_positivos = [
        (posicao, doc_id, score)
        for posicao, (doc_id, score) in enumerate(
            ranking[:TOPO],
            start=1
        )
        if doc_id not in docs_relevantes
    ][:N_FALSOS_POSITIVOS]

    relevantes_omitidos = [
        (posicao, doc_id, score)
        for posicao, (doc_id, score) in enumerate(
            ranking[TOP_K:],
            start=TOP_K + 1
        )
        if doc_id in docs_relevantes
    ][:N_RELEVANTES_OMITIDOS]

    return falsos_positivos, relevantes_omitidos


def montar_tabela(
    query_id,
    itens,
    falsos=None,
    omitidos=None
):
    falsos = set() if falsos is None else falsos
    omitidos = set() if omitidos is None else omitidos

    docs_relevantes = relevantes.get(query_id, set())
    linhas = []

    for posicao, doc_id, score in itens:
        dados = documentos[doc_id]

        if doc_id in falsos:
            categoria = "Falso positivo selecionado"
        elif doc_id in omitidos:
            categoria = "Relevante fora do Top-10"
        else:
            categoria = ""

        linhas.append({
            "Posição": posicao,
            "Doc.": doc_id,
            "Score": score,
            "Relevante": (
                "Sim"
                if doc_id in docs_relevantes
                else "Não"
            ),
            "Categoria": categoria,
            "Título": resumir(
                dados["titulo"],
                100
            ),
            "Texto": resumir(
                dados["texto"],
                260
            )
        })

    return pd.DataFrame(linhas)


def estilo_categoria(linha):
    categoria = linha.get("Categoria", "")

    if categoria == "Falso positivo selecionado":
        estilo = (
            "background-color: #f8d7da;"
            "color: #721c24;"
            "font-weight: bold;"
            "border-top: 2px solid #dc3545;"
            "border-bottom: 2px solid #dc3545;"
        )

        return [estilo] * len(linha)

    if categoria == "Relevante fora do Top-10":
        estilo = (
            "background-color: #d4edda;"
            "color: #155724;"
            "font-weight: bold;"
            "border-top: 2px solid #28a745;"
            "border-bottom: 2px solid #28a745;"
        )

        return [estilo] * len(linha)

    return [""] * len(linha)


def estilizar(tabela, formatos=None):
    resultado = (
        tabela.style
        .apply(
            estilo_categoria,
            axis=1
        )
        .set_properties(**{
            "text-align": "left",
            "vertical-align": "top"
        })
        .set_table_styles([{
            "selector": "th",
            "props": [
                ("background-color", "#343a40"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "left")
            ]
        }])
    )

    if formatos:
        resultado = resultado.format(formatos)

    return resultado


def diagnostico_vetorial(
    query_id,
    doc_id,
    categoria
):
    texto_query = consultas[query_id]["texto"]
    doc_dataset_id = documentos[doc_id]["id_dataset"]

    frequencias_query = Counter(
        preprocess(
            texto_query,
            preprocessing_type
        )
    )

    N = inverted_index.n_documents
    produto_escalar = 0.0
    soma_quadrados_query = 0.0
    linhas = []

    for termo, tf_query in frequencias_query.items():
        postings = inverted_index.index.get(
            termo,
            {}
        )

        df = len(postings)

        if df > 0:
            idf = math.log(N / df)

            peso_query = (
                1 + math.log(tf_query)
            ) * idf

            tf_documento = postings.get(
                doc_dataset_id,
                0
            )

            peso_documento = (
                modelo_vetorial
                .weights[termo]
                .get(doc_dataset_id, 0.0)
            )
        else:
            idf = 0.0
            peso_query = 0.0
            tf_documento = 0
            peso_documento = 0.0

        contribuicao = (
            peso_query * peso_documento
        )

        produto_escalar += contribuicao
        soma_quadrados_query += peso_query ** 2

        linhas.append({
            "Categoria": categoria,
            "Doc.": doc_id,
            "Termo": termo,
            "TF query": tf_query,
            "TF documento": tf_documento,
            "DF": df,
            "N": N,
            "IDF": idf,
            "Peso query": peso_query,
            "Peso documento": peso_documento,
            "Produto": contribuicao
        })

    norma_query = math.sqrt(
        soma_quadrados_query
    )

    norma_documento = (
        modelo_vetorial
        .document_norm
        .get(doc_dataset_id, 0.0)
    )

    score_calculado = (
        produto_escalar
        / (norma_query * norma_documento)
        if norma_query > 0 and norma_documento > 0
        else 0.0
    )

    score_modelo = modelo_vetorial.score(
        texto_query,
        doc_dataset_id,
        preprocessing_type=preprocessing_type
    )

    resumo = {
        "Categoria": categoria,
        "Doc.": doc_id,
        "N": N,
        "Norma query": norma_query,
        "Norma documento": norma_documento,
        "Produto escalar": produto_escalar,
        "Score recalculado": score_calculado,
        "Score do modelo": score_modelo
    }

    return resumo, linhas


def diagnostico_bm25(
    query_id,
    doc_id,
    categoria
):
    texto_query = consultas[query_id]["texto"]
    doc_dataset_id = documentos[doc_id]["id_dataset"]

    frequencias_query = Counter(
        preprocess(
            texto_query,
            preprocessing_type
        )
    )

    N = inverted_index.n_documents
    k1 = bm25.k1
    b = bm25.b
    dl = inverted_index.document_length[
        doc_dataset_id
    ]
    avgdl = inverted_index.avgdl

    score_calculado = 0.0
    linhas = []

    for termo, qtf in frequencias_query.items():
        postings = inverted_index.index.get(
            termo,
            {}
        )

        df = len(postings)
        tf = postings.get(doc_dataset_id, 0)

        if df > 0:
            idf = math.log(
                1 + (
                    (N - df + 0.5)
                    / (df + 0.5)
                )
            )
        else:
            idf = 0.0

        denominador = (
            tf
            + k1 * (
                1
                - b
                + b * (dl / avgdl)
            )
        )

        fator_tf = (
            (tf * (k1 + 1))
            / denominador
            if tf > 0 and denominador > 0
            else 0.0
        )

        contribuicao_unitaria = (
            idf * fator_tf
        )

        contribuicao_total = (
            qtf * contribuicao_unitaria
        )

        score_calculado += contribuicao_total

        linhas.append({
            "Categoria": categoria,
            "Doc.": doc_id,
            "Termo": termo,
            "QTF": qtf,
            "TF documento": tf,
            "DF": df,
            "N": N,
            "IDF BM25": idf,
            "k1": k1,
            "b": b,
            "DL": dl,
            "AvgDL": avgdl,
            "Fator TF": fator_tf,
            "Contribuição unitária": (
                contribuicao_unitaria
            ),
            "Contribuição total": (
                contribuicao_total
            )
        })

    score_modelo = bm25.score(
        texto_query,
        doc_dataset_id,
        preprocessing_type=preprocessing_type
    )

    resumo = {
        "Categoria": categoria,
        "Doc.": doc_id,
        "N": N,
        "k1": k1,
        "b": b,
        "DL": dl,
        "AvgDL": avgdl,
        "Score recalculado": score_calculado,
        "Score do modelo": score_modelo
    }

    return resumo, linhas


def exibir_calculos(
    query_id,
    nome_modelo,
    selecionados
):
    resumos = []
    detalhes = []

    ordem_documentos = {
        doc_id: ordem
        for ordem, (
            _,
            _,
            doc_id,
            _
        ) in enumerate(selecionados)
    }

    for categoria, _, doc_id, _ in selecionados:
        if nome_modelo == "Modelo Vetorial":
            resumo, termos = diagnostico_vetorial(
                query_id,
                doc_id,
                categoria
            )
        else:
            resumo, termos = diagnostico_bm25(
                query_id,
                doc_id,
                categoria
            )

        resumos.append(resumo)
        detalhes.extend(termos)

    tabela_resumo = pd.DataFrame(resumos)
    tabela_detalhes = pd.DataFrame(detalhes)

    tabela_resumo["_ordem"] = (
        tabela_resumo["Doc."]
        .map(ordem_documentos)
    )

    tabela_resumo = (
        tabela_resumo
        .sort_values("_ordem")
        .drop(columns="_ordem")
        .reset_index(drop=True)
    )

    print("\nRESUMO DO CÁLCULO DOS SCORES")

    if nome_modelo == "Modelo Vetorial":
        formatos_resumo = {
            "Norma query": "{:.6f}",
            "Norma documento": "{:.6f}",
            "Produto escalar": "{:.6f}",
            "Score recalculado": "{:.6f}",
            "Score do modelo": "{:.6f}"
        }
    else:
        formatos_resumo = {
            "k1": "{:.2f}",
            "b": "{:.2f}",
            "AvgDL": "{:.2f}",
            "Score recalculado": "{:.6f}",
            "Score do modelo": "{:.6f}"
        }

    estilo_resumo = estilizar(
        tabela_resumo,
        formatos_resumo
    )

    estilo_resumo = (
        estilo_resumo
        .bar(
            subset=["Score recalculado"],
            color="#7aa6c2"
        )
        .bar(
            subset=["Score do modelo"],
            color="#4c78a8"
        )
    )

    display(estilo_resumo)

    if nome_modelo == "Modelo Vetorial":
        coluna_contribuicao = "Produto"
        coluna_idf = "IDF"
        coluna_percentual = "Impacto no produto (%)"

        print(
            "\nFórmula: IDF = ln(N/DF); "
            "peso = (1 + ln(TF)) × IDF; "
            "score = produto escalar / "
            "(norma da query × norma do documento)."
        )

        formatos_detalhes = {
            "IDF": "{:.6f}",
            "Peso query": "{:.6f}",
            "Peso documento": "{:.6f}",
            "Produto": "{:.6f}",
            coluna_percentual: "{:.2f}%"
        }

    else:
        coluna_contribuicao = "Contribuição total"
        coluna_idf = "IDF BM25"
        coluna_percentual = "Impacto no score (%)"

        print(
            "\nFórmula: IDF = ln(1 + "
            "(N − DF + 0,5)/(DF + 0,5)); "
            "contribuição = IDF × "
            "[TF × (k1 + 1)] / "
            "[TF + k1 × "
            "(1 − b + b × DL/AvgDL)]."
        )

        formatos_detalhes = {
            "IDF BM25": "{:.6f}",
            "k1": "{:.2f}",
            "b": "{:.2f}",
            "AvgDL": "{:.2f}",
            "Fator TF": "{:.6f}",
            "Contribuição unitária": "{:.6f}",
            "Contribuição total": "{:.6f}",
            coluna_percentual: "{:.2f}%"
        }

    tabela_detalhes["Magnitude"] = (
        tabela_detalhes[
            coluna_contribuicao
        ].abs()
    )

    totais_por_documento = (
        tabela_detalhes
        .groupby("Doc.")["Magnitude"]
        .transform("sum")
    )

    tabela_detalhes[coluna_percentual] = [
        (
            100 * magnitude / total
            if total > 0
            else 0.0
        )
        for magnitude, total in zip(
            tabela_detalhes["Magnitude"],
            totais_por_documento
        )
    ]

    tabela_detalhes["Ordem do termo"] = (
        tabela_detalhes
        .groupby("Doc.")["Magnitude"]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

    tabela_detalhes["_ordem_documento"] = (
        tabela_detalhes["Doc."]
        .map(ordem_documentos)
    )

    tabela_detalhes = (
        tabela_detalhes
        .sort_values([
            "_ordem_documento",
            "Ordem do termo"
        ])
        .drop(columns=[
            "_ordem_documento",
            "Magnitude"
        ])
        .reset_index(drop=True)
    )

    colunas_iniciais = [
        "Categoria",
        "Doc.",
        "Ordem do termo",
        "Termo",
        coluna_percentual
    ]

    outras_colunas = [
        coluna
        for coluna in tabela_detalhes.columns
        if coluna not in colunas_iniciais
    ]

    tabela_detalhes = tabela_detalhes[
        colunas_iniciais + outras_colunas
    ]

    print(
        "\nCONTRIBUIÇÃO DOS TERMOS "
        "EM ORDEM DE IMPORTÂNCIA"
    )

    print(
        "Os termos estão ordenados da maior para "
        "a menor contribuição dentro de cada documento."
    )

    print(
        "\nBarras azuis: percentual explicado pelo termo."
        "\nBarras amarelas: magnitude do IDF."
        "\nBarras roxas: contribuição efetiva."
    )

    estilo_detalhes = estilizar(
        tabela_detalhes,
        formatos_detalhes
    )

    estilo_detalhes = (
        estilo_detalhes
        .bar(
            subset=[coluna_percentual],
            color="#5b9bd5",
            vmin=0,
            vmax=100
        )
        .bar(
            subset=[coluna_idf],
            color="#f2cf5b",
            vmin=0
        )
        .bar(
            subset=[coluna_contribuicao],
            color="#9c7bd8",
            vmin=0
        )
    )

    display(estilo_detalhes)


def exibir_modelo(query_id, nome_modelo):
    ranking = calcular_ranking(
        query_id,
        nome_modelo
    )

    falsos, omitidos = selecionar_erros(
        query_id,
        ranking
    )

    ids_falsos = {
        doc_id
        for _, doc_id, _ in falsos
    }

    ids_omitidos = {
        doc_id
        for _, doc_id, _ in omitidos
    }

    top10 = [
        (posicao, doc_id, score)
        for posicao, (doc_id, score) in enumerate(
            ranking[:TOP_K],
            start=1
        )
    ]

    print("\n" + "-" * 120)
    print(nome_modelo.upper())
    print("-" * 120)

    print("\nTOP-10")

    print(
        "Vermelho: dois primeiros documentos "
        f"não relevantes encontrados no Top-{TOPO}."
    )

    tabela_top10 = montar_tabela(
        query_id,
        top10,
        falsos=ids_falsos
    )

    display(
        estilizar(
            tabela_top10,
            {"Score": "{:.6f}"}
        )
    )

    print("\nRELEVANTES FORA DO TOP-10")

    print(
        "Verde: dois documentos relevantes mais "
        "bem posicionados depois do Top-10."
    )

    tabela_omitidos = montar_tabela(
        query_id,
        omitidos,
        omitidos=ids_omitidos
    )

    display(
        estilizar(
            tabela_omitidos,
            {"Score": "{:.6f}"}
        )
    )

    selecionados = [
        (
            "Falso positivo selecionado",
            posicao,
            doc_id,
            score
        )
        for posicao, doc_id, score in falsos
    ]

    selecionados += [
        (
            "Relevante fora do Top-10",
            posicao,
            doc_id,
            score
        )
        for posicao, doc_id, score in omitidos
    ]

    exibir_calculos(
        query_id,
        nome_modelo,
        selecionados
    )


def exibir_query(query_id):
    query_id = normalizar_id(query_id)
    texto_query = consultas[query_id]["texto"]

    print("\n\n" + "=" * 120)
    print(f"QUERY ORIGINAL {query_id}")
    print("=" * 120)

    print("\nTEXTO DA QUERY:")
    print(texto_query)

    print(
        f"\nDocumentos relevantes segundo os qrels: "
        f"{len(relevantes.get(query_id, set()))}"
    )

    print(
        "\nLegenda:"
        "\n- Vermelho: falso positivo selecionado."
        "\n- Verde: relevante fora do Top-10."
    )

    for nome_modelo in modelos:
        exibir_modelo(
            query_id,
            nome_modelo
        )


print("=" * 120)
print("9 — ANÁLISE DE ERROS")
print("=" * 120)

for query_id in QUERIES:
    exibir_query(query_id)

9 — ANÁLISE DE ERROS


QUERY ORIGINAL 57

TEXTO DA QUERY:
what are the significant steady and non-steady flow characteristics
which affect the flutter mechanism .

Documentos relevantes segundo os qrels: 14

Legenda:
- Vermelho: falso positivo selecionado.
- Verde: relevante fora do Top-10.

------------------------------------------------------------------------------------------------------------------------
MODELO VETORIAL
------------------------------------------------------------------------------------------------------------------------

TOP-10
Vermelho: dois primeiros documentos não relevantes encontrados no Top-5.


AttributeError: The '.style' accessor requires jinja2